# ZETARIX Phase 3 — QLoRA Fine-tune (Colab)

Fine-tune **Law Interpreter** and **Tag Generator** as separate LoRA adapters on Llama 3 8B.

**Before running:** upload formatted chat JSONL from the repo:
- `backend/data/training/formatted/law_interpreter_train_chat.jsonl`
- `backend/data/training/formatted/law_interpreter_val_chat.jsonl`
- (repeat for `tag_generator_*` when training the second adapter)

Regenerate locally: `PYTHONPATH=src python -m zetarix.training.finetune --format-only`

In [ ]:
# Runtime: GPU (T4/L4/A100)
STAGE = "law_interpreter"  # or "tag_generator"
TRAIN_FILE = f"{STAGE}_train_chat.jsonl"
VAL_FILE = f"{STAGE}_val_chat.jsonl"

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets transformers

In [ ]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_SEQ_LENGTH = 1024
LORA_RANK = 16
LORA_ALPHA = 32
LR = 2e-4
EPOCHS = 3

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=10,
        eval_strategy="epoch",
        output_dir=f"lora-{STAGE}",
        report_to="none",
    ),
)
trainer.train()
model.save_pretrained(f"adapter-{STAGE}")
tokenizer.save_pretrained(f"adapter-{STAGE}")

In [ ]:
# Export merged GGUF for Ollama (q4_k_m — fits 8 GB VRAM inference)
model, tokenizer = FastLanguageModel.from_pretrained(f"adapter-{STAGE}", max_seq_length=MAX_SEQ_LENGTH)
model.save_pretrained_gguf(f"zetarix-{STAGE}", tokenizer, quantization_method="q4_k_m")

## Deploy on your machine

1. Download `zetarix-{stage}-*.gguf` and `Modelfile.{stage}` from `python -m zetarix.training.finetune --export-ollama`
2. `ollama create zetarix-law-interpreter:latest -f Modelfile.law_interpreter`
3. ```bash
export ZETARIX_LLM_BACKEND=local
export OLLAMA_MODEL_LAW_INTERPRETER=zetarix-law-interpreter:latest
export OLLAMA_MODEL_TAG_GENERATOR=zetarix-tag-generator:latest
```